# Lab 03: Bedrock Agent — Conversational Certificate Management

In this lab you will:
1. Create a Bedrock Agent with the CertAgent system prompt
2. Attach the Lambda action group using the OpenAPI schema
3. Have natural language conversations that invoke real Lambda functions
4. Inspect agent reasoning traces

```
You: "Renew all critical certs"
  -> Bedrock Agent (Claude) -> Lambda scan -> Lambda renew -> summary
```

> **Estimated time:** 30 minutes

## Load config

In [ ]:
# Config is written by the SageMaker lifecycle script from SSM at space startup.
# If this fails, re-launch the JupyterLab space to trigger the lifecycle script.
import json, pathlib, boto3

config = json.loads(pathlib.Path('/tmp/certagent_config.json').read_text())
globals().update(config)

REPO_DIR = pathlib.Path(REPO_DIR)
lm  = boto3.client('lambda',        region_name=AWS_REGION)
ddb = boto3.resource('dynamodb',    region_name=AWS_REGION)
sm  = boto3.client('secretsmanager',region_name=AWS_REGION)

PRIORITY_EMOJI = {'EXPIRED': '💀', 'CRITICAL': '🔴', 'HIGH': '🟠', 'MEDIUM': '🟡', 'LOW': '🟢'}

def invoke(fn, payload):
    r = lm.invoke(FunctionName=fn, InvocationType='RequestResponse',
                  Payload=json.dumps(payload))
    raw = json.loads(r['Payload'].read())
    if 'FunctionError' in r:
        raise RuntimeError(raw.get('errorMessage', raw))
    return raw.get('body', raw)

print(f'Region  : {AWS_REGION}')
print(f'Table   : {CERT_TABLE_NAME}')
print(f'Lambda  : {LAMBDA_SCAN}')
print('✅ Environment ready')

In [ ]:
import time, uuid
bedrock_agent_client = boto3.client('bedrock-agent',         region_name=AWS_REGION)
bedrock_runtime      = boto3.client('bedrock-agent-runtime', region_name=AWS_REGION)
iam                  = boto3.client('iam')
sts                  = boto3.client('sts')
ACCOUNT_ID = sts.get_caller_identity()['Account']
AGENT_NAME = 'CertAgent'
ALIAS_NAME = 'workshop'
MODEL_ID   = 'anthropic.claude-3-sonnet-20240229-v1:0'
print(f'Account: {ACCOUNT_ID}')

## Read agent instruction & API schema

In [ ]:
agent_instruction = (REPO_DIR / 'agent' / 'agent_instruction.txt').read_text()
api_schema_text   = (REPO_DIR / 'agent' / 'api_schema.yaml').read_text()
print(agent_instruction[:500])

## Create IAM role for the agent

In [ ]:
ROLE_NAME = f'{WORKSHOP_PREFIX}-bedrock-agent-role'
trust = {'Version':'2012-10-17','Statement':[{'Effect':'Allow',
    'Principal':{'Service':'bedrock.amazonaws.com'},'Action':'sts:AssumeRole',
    'Condition':{'StringEquals':{'aws:SourceAccount':ACCOUNT_ID}}}]}
try:
    role = iam.create_role(RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust))
    ROLE_ARN = role['Role']['Arn']
    print(f'Created: {ROLE_ARN}')
except iam.exceptions.EntityAlreadyExistsException:
    ROLE_ARN = iam.get_role(RoleName=ROLE_NAME)['Role']['Arn']
    print(f'Exists: {ROLE_ARN}')

pol = {'Version':'2012-10-17','Statement':[
    {'Effect':'Allow','Action':['bedrock:InvokeModel','bedrock:InvokeModelWithResponseStream'],
     'Resource':f'arn:aws:bedrock:{AWS_REGION}::foundation-model/*'},
    {'Effect':'Allow','Action':'lambda:InvokeFunction',
     'Resource':f'arn:aws:lambda:{AWS_REGION}:{ACCOUNT_ID}:function:{WORKSHOP_PREFIX}-*'}]}
iam.put_role_policy(RoleName=ROLE_NAME, PolicyName='AgentPolicy',
    PolicyDocument=json.dumps(pol))
print('Policy attached')
time.sleep(10)

## Create the Bedrock Agent

In [ ]:
agents = bedrock_agent_client.list_agents()['agentSummaries']
existing = next((a for a in agents if a['agentName'] == AGENT_NAME), None)
if existing:
    AGENT_ID = existing['agentId']
    print(f'Agent exists: {AGENT_ID}')
else:
    r = bedrock_agent_client.create_agent(agentName=AGENT_NAME,
        agentResourceRoleArn=ROLE_ARN, foundationModel=MODEL_ID,
        instruction=agent_instruction,
        description='Certificate lifecycle AI agent',
        idleSessionTTLInSeconds=1800)
    AGENT_ID = r['agent']['agentId']
    print(f'Created: {AGENT_ID}')
config['AGENT_ID'] = AGENT_ID
pathlib.Path('/tmp/certagent_config.json').write_text(json.dumps(config, indent=2))

## Grant Lambda invoke permissions

In [ ]:
fns = [LAMBDA_SCAN, LAMBDA_RENEW, LAMBDA_STATUS, LAMBDA_DOWNLOAD, LAMBDA_STORE, LAMBDA_INVENTORY]
for fn in fns:
    sid = f'Bedrock{fn.split("-")[-1][:12]}'
    try:
        lm.add_permission(FunctionName=fn, StatementId=sid,
            Action='lambda:InvokeFunction', Principal='bedrock.amazonaws.com',
            SourceArn=f'arn:aws:bedrock:{AWS_REGION}:{ACCOUNT_ID}:agent/{AGENT_ID}')
        print(f'  {fn} ok')
    except lm.exceptions.ResourceConflictException:
        print(f'  {fn} already set')

## Attach action group

In [ ]:
AG_NAME = 'CertAgentActions'
scan_arn = lm.get_function_configuration(FunctionName=LAMBDA_SCAN)['FunctionArn']
ags = bedrock_agent_client.list_agent_action_groups(agentId=AGENT_ID,
    agentVersion='DRAFT')['actionGroupSummaries']
existing_ag = next((a for a in ags if a['actionGroupName'] == AG_NAME), None)
kw = dict(agentId=AGENT_ID, agentVersion='DRAFT', actionGroupName=AG_NAME,
    description='CertAgent tools', actionGroupExecutor={'lambda': scan_arn},
    apiSchema={'payload': api_schema_text}, actionGroupState='ENABLED')
if existing_ag:
    bedrock_agent_client.update_agent_action_group(**kw, actionGroupId=existing_ag['actionGroupId'])
    print('Updated')
else:
    bedrock_agent_client.create_agent_action_group(**kw)
    print('Created')

## Prepare agent & create alias

In [ ]:
bedrock_agent_client.prepare_agent(agentId=AGENT_ID)
for i in range(20):
    st = bedrock_agent_client.get_agent(agentId=AGENT_ID)['agent']['agentStatus']
    if st == 'PREPARED': break
    time.sleep(5)
print(f'Agent status: {st}')

aliases = bedrock_agent_client.list_agent_aliases(agentId=AGENT_ID)['agentAliasSummaries']
ea = next((a for a in aliases if a['agentAliasName'] == ALIAS_NAME), None)
if ea:
    ALIAS_ID = ea['agentAliasId']
else:
    r = bedrock_agent_client.create_agent_alias(agentId=AGENT_ID, agentAliasName=ALIAS_NAME)
    ALIAS_ID = r['agentAlias']['agentAliasId']
config['AGENT_ALIAS_ID'] = ALIAS_ID
pathlib.Path('/tmp/certagent_config.json').write_text(json.dumps(config, indent=2))
print(f'Alias: {ALIAS_ID}')

## Populate inventory

In [ ]:
invoke(LAMBDA_SCAN, {'use_mock': True, 'threshold_days': 60})
print('Done')

## Talk to CertAgent

In [ ]:
def ask(msg, sid=None):
    sid = sid or f'w-{uuid.uuid4().hex[:8]}'
    print(f'You: {msg}'); print('-'*60)
    r = bedrock_runtime.invoke_agent(agentId=AGENT_ID, agentAliasId=ALIAS_ID,
        sessionId=sid, inputText=msg)
    txt = ''
    for ev in r['completion']:
        if 'chunk' in ev:
            c_ = ev['chunk']['bytes'].decode(); txt += c_
            print(c_, end='', flush=True)
    print('\n'+'='*60)
    return txt, sid
print('ask() ready')

In [ ]:
r, sid = ask('What certs are expiring soon? Prioritized summary please.')

In [ ]:
r, sid = ask('Renew api.example.com using mock mode.', sid)

In [ ]:
r, sid = ask('Yes proceed.', sid)

In [ ]:
r, sid = ask('Scan for certs expiring in 14 days with mock data, '
    'then renew any CRITICAL or EXPIRED ones.')

In [ ]:
r, sid = ask('Show full inventory grouped by status.', sid)

## Inspect traces

In [ ]:
tr = bedrock_runtime.invoke_agent(agentId=AGENT_ID, agentAliasId=ALIAS_ID,
    sessionId=f'tr-{uuid.uuid4().hex[:6]}',
    inputText='What certs expire in 7 days? Use mock.', enableTrace=True)
for ev in tr['completion']:
    if 'trace' in ev:
        ot = ev['trace'].get('trace',{}).get('orchestrationTrace',{})
        if 'rationale' in ot: print('🧠', ot['rationale']['text'][:200])
        if 'invocationInput' in ot:
            ai = ot['invocationInput'].get('actionGroupInvocationInput',{})
            if ai: print(f'🔧 {ai.get("function")} {ai.get("parameters")}')
    if 'chunk' in ev: print('💬', ev['chunk']['bytes'].decode()[:300])

## Lab 03 Complete

**Next:** `04_proactive_monitoring.ipynb`